# Session 07, Notebook 1: trace the Research Brief Agent with Langfuse

**Goal.** Send every run of the agent to Langfuse, read the trace in the UI, then add the data you need for analysis.

- **Part A, automatic tracing:** one LangChain `CallbackHandler` records every graph step, model call and tool call.
- **Part B, custom tracing:** a named root observation, trace attributes (`user_id`, `session_id`, `tags`, `metadata`), the prompt
  version, an image in the user message and a user-feedback score.

<img src="images/b7_trace_flow.png" alt="callback events become observations, the SDK sends them in batches" width="640">

Before you start: create the prompt `research-brief` in Langfuse by copying `prompts/research-brief_v1.txt` and `prompts/research-brief_v2.txt` (README, step 5).
Docs: [LangChain integration](https://langfuse.com/integrations/frameworks/langchain) ·
[Data model](https://langfuse.com/docs/observability/data-model) ·
[Best practices](https://langfuse.com/docs/observability/best-practices) ·
[Multi-modality](https://langfuse.com/docs/observability/features/multi-modality)

In [1]:
import logging, os, sys

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
for noisy in ("httpx", "httpx2"):  # HTTP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
logger = logging.getLogger(__name__)

from dotenv import load_dotenv
from langfuse import get_client

load_dotenv("../.env")
sys.path.insert(0, "..")                     # the session folder, for app/tools.py
for key in ("OPENAI_API_KEY", "TAVILY_API_KEY", "MODEL", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"):
    if not os.getenv(key):
        raise SystemExit(f"{key} is missing: copy .env.example to .env and fill it in (README, step 3)")
langfuse = get_client()
if not langfuse.auth_check():
    raise SystemExit("Langfuse rejected the keys: check LANGFUSE_* in .env and that LANGFUSE_BASE_URL is your region")
logger.info("Langfuse connected | model: %s", os.getenv("MODEL"))

INFO | Langfuse connected | model: openai:gpt-6-luna


## 1. The agent, with its prompt from Langfuse
Three tools from `app/tools.py`: `search_kb` (a small knowledge base), `search_docs` (web search on docs.langchain.com) and `read_doc`
(reads one page, cached so every run reads the same text). The system prompt is **not** in the code: `get_prompt` fetches the version
labelled `v1` from Langfuse Prompt Management (the SDK caches it for 60 seconds).

In [2]:
def get_prompt(name, label="production"):
    """Fetch a prompt created in the Langfuse UI (README, step 5); stop with a clear message if it is missing."""
    try:
        return langfuse.get_prompt(name, label=label)
    except Exception as exc:
        raise SystemExit(f"Prompt {name!r} with label {label!r} not found in Langfuse: create it (README, step 5)") from exc

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from app.tools import TOOLS, UserContext

prompt = get_prompt("research-brief", label="v1")
agent = create_agent(init_chat_model(os.environ["MODEL"]), TOOLS, system_prompt=prompt.compile(), context_schema=UserContext)
print(f"prompt {prompt.name}, version {prompt.version}, labels {prompt.labels}")

prompt research-brief, version 1, labels ['v1']


## Part A. Automatic tracing: one callback handler
`CallbackHandler()` is a LangChain **callback**: LangChain calls it when a graph step, a model call or a tool call starts, ends or fails,
and the handler turns each event into a Langfuse **observation**. Passing it in `config={"callbacks": [...]}` is enough; nested steps land
in the same **trace**. Traces are sent in the background, so `flush()` before you open the link.

In [3]:
from langfuse.langchain import CallbackHandler

question = "How does a LangGraph checkpointer save the state of a run?"
handler = CallbackHandler()
result = agent.invoke({"messages": [("user", question)]}, config={"callbacks": [handler]}, context=UserContext("alice"))
langfuse.flush()
print(result["messages"][-1].text[:500])
print("\ntrace:", langfuse.get_trace_url(trace_id=handler.last_trace_id))

INFO | search_kb('LangGraph checkpointer how saves state of a run checkpoints threads snapshots state')
INFO | read_doc(langgraph/checkpointers)
INFO | read_doc(langgraph/persistence)
INFO | read_doc(langgraph/use-time-travel)


A LangGraph checkpointer saves **snapshots of graph state as the graph runs**, typically at each super-step (a round of node execution). Each snapshot records the state values and what should run next, along with execution metadata such as node writes and the step number. [Checkpointers](https://docs.langchain.com/oss/python/langgraph/checkpointers)

Snapshots are associated with a **thread**, identified by `thread_id` in the graph’s config. That lets later invocations continue or inspect the sa

trace: https://us.cloud.langfuse.com/project/cmumwadzp031sad0c12u9bfvq/traces/d0869bfd9d2591ab23de34c260a80462


Open the trace. **Recorded by itself:** the graph run and its nodes (`CHAIN`), every model call (`GENERATION`: messages, tokens, cost,
model), every tool call (`TOOL`: arguments, result), durations and errors. The `RETRIEVER retrieve-kb-pages` line comes from one decorator
in `app/tools.py`, because plain Python code fires no LangChain callback:
```python
@observe(as_type="retriever", name="retrieve-kb-pages")
def retrieve(query, product, team, k=3): ...
```
**Missing for analysis:** who asked (`user_id`), which conversation (`session_id`), which prompt version, and a trace name that says what the run did.

## Part B. Custom tracing: add what you want to analyse
- `start_as_current_observation(name="write-brief", as_type="agent")` opens a **root observation**: one run = one trace with a clear name,
  input and output. A `CallbackHandler` created inside it joins the same trace.
- `propagate_attributes(user_id=..., session_id=..., tags=[...], metadata={...}, prompt=prompt)` puts **trace attributes** on every observation
  of the block and links every model call to the **prompt version** (Prompts → research-brief → **Metrics** counts them per version).

In [4]:
import base64, uuid
from pathlib import Path
from langfuse import propagate_attributes

SESSION_ID = f"brief-{uuid.uuid4().hex[:6]}"   # one conversation = one session id


def ask(question, user_id="alice", image_path=None):
    """One traced run of the agent. Returns (answer, trace_id)."""
    content = question
    if image_path:  # an image in the user message: the SDK uploads it and keeps a reference in the trace
        data = base64.b64encode(Path(image_path).read_bytes()).decode()
        content = [{"type": "text", "text": question}, {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{data}"}}]
    with langfuse.start_as_current_observation(name="write-brief", as_type="agent", input={"question": question}) as root:
        with propagate_attributes(user_id=user_id, session_id=SESSION_ID, tags=["lab07", "prompt-v1"],
                                  metadata={"team": UserContext(user_id).team}, prompt=prompt):
            result = agent.invoke({"messages": [("user", content)]}, config={"callbacks": [CallbackHandler()]},
                                  context=UserContext(user_id))
        answer = result["messages"][-1].text
        root.update(output=answer)
    return answer, root.trace_id


answer, trace_b = ask(question)
langfuse.flush()
print(answer[:300])
print("\ntrace:", langfuse.get_trace_url(trace_id=trace_b))

INFO | search_kb('LangGraph checkpointer how saves graph state run checkpoints thread_id state snapshot')
INFO | read_doc(langgraph/checkpointers)
INFO | search_kb('LangGraph checkpoint saves channels pending writes super-step checkpoint storage thread checkpoint_id serialization saver')
INFO | read_doc(langgraph/persistence)
INFO | search_docs('site:docs.langchain.com/oss/python/langgraph checkpointer checkpoint each super-step pending writes')
INFO | read_doc(langgraph/graph-api)


A checkpointer records **graph-state snapshots as execution progresses**, typically at the end of each super-step. A super-step is one graph execution round; the snapshot captures the state at that point, rather than merely saving the final output. [Checkpointers](https://docs.langchain.com/oss/pyth

trace: https://us.cloud.langfuse.com/project/cmumwadzp031sad0c12u9bfvq/traces/90fdff401bc6e6d0436f9e3250b81c97


Compare with Part A: the root is now `AGENT write-brief` with the question as input and the brief as output; `user_id`, `session_id` and
`tags` are set; each `GENERATION` shows the prompt `research-brief` version 1.

### Two more questions in the same session, one out of scope
A **session** groups the traces of one conversation (**Tracing → Sessions**). The weather question is outside the docs: a good agent says
so **without** calling tools. Look at what v1 does.

In [5]:
for q in ["Now explain what a thread is in LangGraph.", "What is the weather in Hanoi tomorrow?"]:
    answer, _ = ask(q)
    print(f"{q}\n-> {answer[:200]}\n")
langfuse.flush()
print("session:", SESSION_ID)

INFO | search_kb('LangGraph thread definition persistence conversation identifier checkpoints thread_id')
INFO | read_doc(langgraph/persistence)
INFO | search_docs('site:docs.langchain.com/oss/python/langgraph thread_id threads checkpoints thread')
INFO | read_doc(langgraph/checkpointers)
INFO | read_doc(langgraph/overview)


Now explain what a thread is in LangGraph.
-> In LangGraph, a **thread** is a persistent sequence of graph state associated with a `thread_id`. You pass that ID in the graph’s configuration, for example `{"configurable": {"thread_id": "thread-1"}

What is the weather in Hanoi tomorrow?
-> I can’t access live weather forecasts, so I can’t reliably tell you Hanoi’s weather tomorrow. Please check a weather service for the latest forecast.

session: brief-aa3b6d


### An image in the user message
Put the image in the **user message**: the SDK finds the base64 data, uploads it to Langfuse and keeps a reference, so the generation
shows the image instead of a long string. The image is the checkpoints figure from the LangGraph docs.

In [6]:
answer, trace_img = ask("A teammate sent me this diagram from the LangGraph docs. What does it show? Write a short brief with links.",
                        image_path="../data/checkpoints_diagram.png")
langfuse.flush()
print(answer[:300])
print("\ntrace:", langfuse.get_trace_url(trace_id=trace_img))

INFO | search_kb('LangGraph persistence checkpoints threads super-steps StateSnapshot graph nodes edges diagram')
INFO | read_doc(langgraph/persistence)
INFO | read_doc(langgraph/graph-api)
INFO | read_doc(langgraph/use-subgraphs)
INFO | search_kb('LangGraph StateSnapshot fields values next config metadata created_at checkpoint thread_id checkpoint docs super-step parallel nodes')
INFO | read_doc(langgraph/checkpointers)
INFO | read_doc(langgraph/use-time-travel)


The diagram connects LangGraph’s execution flow to its persistence model. The top row shows state moving through two nodes: the graph begins with `{"state": "I"}` and ends with `{"state": "I heart langgraph"}`. Nodes and edges define that flow. [Graph API overview](https://docs.langchain.com/oss/pyt

trace: https://us.cloud.langfuse.com/project/cmumwadzp031sad0c12u9bfvq/traces/6a7deb5eba1c422629659ef57ff41c2e


### User feedback as a score
A thumbs up or down from your app is a **score** on the trace. Evaluators in Notebook 2 also write scores, so feedback and automatic
checks sit side by side (**Scores** tab of the trace).

In [7]:
langfuse.create_score(trace_id=trace_b, name="user_feedback", value=1, data_type="BOOLEAN", comment="clear brief, links work")
langfuse.flush()

## 2. Write down what failed
Open the traces of this notebook and write 2 or 3 lines: which step was wrong, which tool was called that should not have been, which url
was cited. These notes become eval cases in Notebook 2.

*Example from the tested run:* for the weather question v1 called `search_docs`, read pages unrelated to weather and cited them.